# **Condition Chains**

In [43]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableSequence, RunnableLambda, RunnableParallel, RunnableBranch

from dotenv import load_dotenv
import os
load_dotenv()


if os.environ['GEMINI_API_KEY']:
    print('gemini key set')
else:
    print("not set")

gemini key set


In [44]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)

In [45]:
from pydantic import Field, BaseModel
from typing import Literal

class llm_review_schema(BaseModel):
    movie_summary_flag: Literal['Positive','Negative'] = Field(description="Return whether a movie review is negative or positive")


llm_structured_output = llm.with_structured_output(llm_review_schema)

# demo_chain = llm_structured_output

# demo_chain.invoke("This movie is trash")




# **Chain With Conditional Chains**

In [46]:

# Task 1, [Prompt]
prompt_template = ChatPromptTemplate.from_messages([
    ("system","You are a movie review evaluator"),
    ("user", "Please categorize the movie review as either positive or negative: {input}")
])


In [47]:
# Task 3, [String Parser]
str_parser = StrOutputParser()

In [48]:
# Task 4, [Custom Runnable]

def pydantic_json(input:llm_review_schema) -> str:
    return input.model_dump()['movie_summary_flag']


pydantic_runnable_lambda = RunnableLambda(pydantic_json)

## **Conditional Chain 1**

In [49]:
# Task -1 [Prompt chain 1]

linkedin_prompt = ChatPromptTemplate.from_messages([
    ("system","You're a linkedin post generator"),
    ("user","Create a post for the following text for LinkedIn: {text}")
])

# task -2  chain 1

llm_chain_1 = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_tokens=None,
    timeout=None,
    max_retries=2,
)


# Task 3 string parser chain 1, [String Parser]
str_parser_chain_1 = StrOutputParser()


chain_linkedin = linkedin_prompt | llm_chain_1 | str_parser_chain_1

## **Conditional Chain 2**

In [50]:
def insta_chain(text:dict):
    # Task -1 [Prompt chain 1]

    # text = text['text']

    insta_prompt = ChatPromptTemplate.from_messages([
        ("system","You're an instagram post generator"),
        ("user","Create a post for the following text for instagram: {text}")
    ])

    # task -2  chain 1

    llm_chain_2 = ChatGoogleGenerativeAI(
        model="gemini-3.1-flash-lite",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )


    # Task 3 string parser chain 1, [String Parser]
    str_parser_chain_2 = StrOutputParser()

    chain_insta = insta_prompt | llm_chain_2 | str_parser_chain_2

    # result = chain_insta.invoke(text)

    return chain_insta # result


insta_chain_runnable = RunnableLambda(insta_chain)

## **Final Orchestration**

In [51]:
conditional_chain = RunnableBranch(
    (lambda x: "positive" in x, chain_linkedin),
    insta_chain_runnable
)


final_orchestrator = prompt_template | llm_structured_output | pydantic_runnable_lambda | conditional_chain

final_orchestrator.invoke("Transformers age of extinction is a bad movie")

'To give you the best post, I’ve broken these down by the "vibe" of your account. Choose the one that fits your aesthetic best:\n\n### Option 1: The "Deep & Moody" Vibe (Best for a black-and-white photo)\n**Caption:**\nSometimes, you have to embrace the negative to see the picture clearly. 🎞️🖤\n.\n.\n#mood #perspective #negative #blackandwhite #mindset\n\n### Option 2: The "Short & Punchy" Vibe (Best for a minimalist aesthetic)\n**Caption:**\nNegative. 🎞️\n.\n.\n#minimalist #negative #mood\n\n### Option 3: The "Growth/Inspirational" Vibe (Best for a reflective post)\n**Caption:**\nIn photography, the negative is where the image is born. In life, the "negatives" are often just the space where we grow the most. Change your perspective. 🔄✨\n.\n.\n#growthmindset #perspective #lessons #negative #positivevibes\n\n### Option 4: The "Edgy/Artistic" Vibe (Best for a film/grainy photo)\n**Caption:**\nDeveloping. 🎞️🌑\n.\n.\n#filmphotography #negative #darkaesthetic #art\n\n---\n\n**💡 Pro-tip for 